In [1]:
import os
import glob
import pandas as pd
import geopandas as gpd
from shapely import geometry
from tqdm import tqdm
import mercantile

In [2]:
### For gloabl

import pandas as pd
import geopandas as gpd
from shapely.geometry import shape, box
import mercantile

# --------------------------------------------------
# 1. Define your study area (anywhere in world)
# --------------------------------------------------

# Bounding box  
min_lon = -92.6006
min_lat = 29.3080
max_lon = -91.5537
max_lat = 31.4420

# max_lat, max_lon = 31.4420,-91.5537
# min_lat, min_lon = 29.3080,-92.6006

# --------------------------------------------------
# 2. Convert extent → AOI polygon
# --------------------------------------------------
aoi_geom = {
    "type": "Polygon",
    "coordinates": [[
        [min_lon, max_lat],
        [min_lon, min_lat],
        [max_lon, min_lat],
        [max_lon, max_lat],
        [min_lon, max_lat]
    ]]
}

aoi_shape = shape(aoi_geom)

# --------------------------------------------------
# 3. Load global building tile index
# --------------------------------------------------
print("Loading global building tile index...")

links = pd.read_csv(
    "https://minedbuildings.z5.web.core.windows.net/global-buildings/dataset-links.csv",
    dtype={"QuadKey": str}
)

# --------------------------------------------------
# 4. Find intersecting tiles
# --------------------------------------------------
print("Finding tiles intersecting AOI...")

selected_tiles = []

for _, row in links.iterrows():

    quadkey = row["QuadKey"]

    tile = mercantile.quadkey_to_tile(quadkey)
    bounds = mercantile.bounds(tile)

    tile_geom = box(bounds.west, bounds.south, bounds.east, bounds.north)

    if tile_geom.intersects(aoi_shape):
        selected_tiles.append(row)

print("Tiles intersecting AOI:", len(selected_tiles))

# --------------------------------------------------
# 5. Download only intersecting tiles
# --------------------------------------------------
buildings = []

for row in selected_tiles:

    print("Downloading tile:", row["QuadKey"])

    df = pd.read_json(row["Url"], lines=True)

    df["geometry"] = df["geometry"].apply(shape)

    gdf = gpd.GeoDataFrame(df, geometry="geometry", crs=4326)

    # clip buildings to AOI
    gdf = gdf[gdf.intersects(aoi_shape)]

    if not gdf.empty:
        buildings.append(gdf)

# --------------------------------------------------
# 6. Merge results
# --------------------------------------------------
if buildings:

    result = gpd.GeoDataFrame(
        pd.concat(buildings, ignore_index=True),
        crs=4326
    )

    result.to_file("data/buildings_t63_IW2.geojson", driver="GeoJSON")

    print("Buildings saved to buildings_aoi.geojson")

else:
    print("No buildings found in AOI")

Loading global building tile index...
Finding tiles intersecting AOI...
Tiles intersecting AOI: 9
Buildings saved to buildings_aoi.geojson


In [3]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
from rasterio.warp import calculate_default_transform, reproject, Resampling
import numpy as np

# ------------------------------------------------
# 1. Load building polygons
# ------------------------------------------------
gdf = gpd.read_file("Khl_buildings_aoi.geojson")

# ------------------------------------------------
# 2. Convert to UTM for accurate meter resolution
# ------------------------------------------------
utm_crs = gdf.estimate_utm_crs()
gdf_utm = gdf.to_crs(utm_crs)

# ------------------------------------------------
# 3. Create 1 m raster grid
# ------------------------------------------------
minx, miny, maxx, maxy = gdf_utm.total_bounds

resolution = 30  # meters

width = int((maxx - minx) / resolution)
height = int((maxy - miny) / resolution)

transform = from_origin(minx, maxy, resolution, resolution)

# ------------------------------------------------
# 4. Rasterize buildings
# ------------------------------------------------
shapes = ((geom, 1) for geom in gdf_utm.geometry)

raster = rasterize(
    shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0,
    dtype="uint8"
)

# ------------------------------------------------
# 5. Save temporary UTM raster
# ------------------------------------------------
utm_raster = "building_30m_utm.tif"

with rasterio.open(
    utm_raster,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype="uint8",
    crs=utm_crs,
    transform=transform,
) as dst:
    dst.write(raster, 1)

# ------------------------------------------------
# 6. Reproject raster to WGS84
# ------------------------------------------------
dst_crs = "EPSG:4326"

with rasterio.open(utm_raster) as src:

    transform, width, height = calculate_default_transform(
        src.crs, dst_crs, src.width, src.height, *src.bounds
    )

    kwargs = src.meta.copy()
    kwargs.update({
        "crs": dst_crs,
        "transform": transform,
        "width": width,
        "height": height
    })

    with rasterio.open("building_30m_wgs84.tif", "w", **kwargs) as dst:

        reproject(
            source=rasterio.band(src, 1),
            destination=rasterio.band(dst, 1),
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=dst_crs,
            resampling=Resampling.nearest
        )

print("Final raster saved: building_30m_wgs84.tif")

Final raster saved: building_30m_wgs84.tif
